# 🛍️ Customer Shopping Behavior Analysis — شرح كامل بالعربي والإنجليزي

**الهدف من النوتبوك (Overall Goal):**
ده notebook بيعمل تحليل أولي (initial analysis / EDA groundwork) لبيانات سلوك شراء العملاء (Customer Shopping Behavior dataset). بشكل عام هنعدي على المراحل دي:

1. **Data Loading** – تحميل البيانات من ملف CSV.
2. **Data Understanding** – نشوف شكل الداتا، الأعمدة، والأنواع (`head`, `info`, `describe`).
3. **Data Quality Check** – فحص القيم الفارغة (missing values).
4. **Data Cleaning** – تعويض القيم الفارغة (imputation) وحذف الأعمدة الزيادة عن الحاجة (redundant columns).
5. **Feature Engineering** – إنشاء أعمدة جديدة مفيدة زي `age_group` و `purchase_frequency_days` عشان تفيد في تحليلات لاحقة زي customer segmentation أو حتى churn prediction.
6. **Data Export** – رفع النسخة النضيفة من الداتا لقاعدة بيانات SQL Server عشان تبقى متاحة لأي أداة تانية (Power BI, Excel, إلخ).

كل خلية جاية بعدين هتلاقي تحتها خلية markdown بتشرح: **إيه اللي بيحصل فيها (What)**، **وليه استخدمناها (Why)**، **والغرض منها (Purpose)**.

In [1]:
# Loading the dataset using pandas

import pandas as pd

df = pd.read_csv('customer_shopping_behavior.csv')

**الشرح (Explanation):** هنا بنعمل import لمكتبة `pandas` — وهي المكتبة الأساسية في بايثون للتعامل مع البيانات الجدولية (tabular data) في شكل `DataFrame`. بعدين بنستخدم `pd.read_csv()` عشان نقرا ملف الـ CSV بتاع بيانات العملاء ونحمّله جوه متغير `df`.

**ليه استخدمناها (Why):** أي تحليل بيانات لازم يبدأ بخطوة تحميل الداتا لبيئة الشغل (environment)، و`pandas` هي أشهر وأقوى أداة لده في بايثون لأنها بتوفر structures زي DataFrame سهلة في الفلترة، التجميع، والتنظيف.

**الغرض (Purpose):** تجهيز الداتا كخطوة أولى قبل أي استكشاف أو تنظيف أو تحليل.

In [2]:
df.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


**الشرح:** `df.head()` بتطبع أول 5 صفوف من الـ DataFrame بشكل افتراضي.

**ليه استخدمناها:** عشان ناخد "نظرة سريعة" (quick preview) على شكل البيانات من غير ما نطبعها كلها — ده أول حاجة بتتعمل في أي EDA (Exploratory Data Analysis) عشان نتعرف على الأعمدة الموجودة (زي Age, Gender, Item Purchased...) ونوع القيم جوه كل عمود.

**الغرض:** التأكد إن البيانات اتحملت صح، وأخد فكرة مبدئية عن الـ structure قبل أي تعديل.

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   str    
 3   Item Purchased          3900 non-null   str    
 4   Category                3900 non-null   str    
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   str    
 7   Size                    3900 non-null   str    
 8   Color                   3900 non-null   str    
 9   Season                  3900 non-null   str    
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   str    
 12  Shipping Type           3900 non-null   str    
 13  Discount Applied        3900 non-null   str    
 14  Promo Code Used         3900 non-null   str    
 15

**الشرح:** `df.info()` بتديلنا ملخص تقني عن الـ DataFrame: عدد الصفوف (3900 row)، عدد وأسماء الأعمدة (20 column)، نوع البيانات (`dtype`) لكل عمود، وعدد القيم الغير فارغة (`Non-Null Count`) لكل عمود.

**ليه استخدمناها:** دي أسرع طريقة نكتشف بيها 3 حاجات مهمة: (1) هل فيه missing values؟ (لاحظ إن `Review Rating` فيها 3863 بس مش 3900)، (2) هل الأنواع (types) صح؟ زي إن الأعمدة الرقمية فعلا `int64`/`float64` مش نص، (3) هل فيه أعمدة زيادة عن الحاجة؟ زي `Unnamed: 18` و `Unnamed: 19` اللي طلعوا فاضيين تمامًا (0 non-null).

**الغرض:** تشخيص سريع (quick diagnostic) لجودة البيانات قبل ما نبدأ التنظيف.

In [4]:
# Summary statistics using .describe()
df.describe(include='all')

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
count,3900.000000,3900.000000,3900,3900,3900,3900.000000,3900,3900,3900,3900,3863.000000,3900,3900,3900,3900,3900.000000,3900,3900
unique,NaN,NaN,2,25,4,NaN,50,4,25,4,NaN,2,6,2,2,NaN,6,7
top,NaN,NaN,Male,Blouse,Clothing,NaN,Montana,M,Olive,Spring,NaN,No,Free Shipping,No,No,NaN,PayPal,Every 3 Months
freq,NaN,NaN,2652,171,1737,NaN,96,1755,177,999,NaN,2847,675,2223,2223,NaN,677,584
mean,1950.500000,44.068462,NaN,NaN,NaN,59.764359,NaN,NaN,NaN,NaN,3.750065,NaN,NaN,NaN,NaN,25.351538,NaN,NaN
std,1125.977353,15.207589,NaN,NaN,NaN,23.685392,NaN,NaN,NaN,NaN,0.716983,NaN,NaN,NaN,NaN,14.447125,NaN,NaN
min,1.000000,18.000000,NaN,NaN,NaN,20.000000,NaN,NaN,NaN,NaN,2.500000,NaN,NaN,NaN,NaN,1.000000,NaN,NaN
25%,975.750000,31.000000,NaN,NaN,NaN,39.000000,NaN,NaN,NaN,NaN,3.100000,NaN,NaN,NaN,NaN,13.000000,NaN,NaN
50%,1950.500000,44.000000,NaN,NaN,NaN,60.000000,NaN,NaN,NaN,NaN,3.800000,NaN,NaN,NaN,NaN,25.000000,NaN,NaN
75%,2925.250000,57.000000,NaN,NaN,NaN,81.000000,NaN,NaN,NaN,NaN,4.400000,NaN,NaN,NaN,NaN,38.000000,NaN,NaN


**الشرح:** `df.describe(include='all')` بتحسب إحصائيات وصفية (summary statistics) لكل الأعمدة. للأعمدة الرقمية (numeric) بتديك `mean`, `std`, `min`, `max`, والـ quartiles (25%, 50%, 75%). وللأعمدة النصية/الفئوية (categorical) بتديك `unique` (عدد القيم المختلفة)، `top` (أكتر قيمة تكرارًا)، و`freq` (عدد تكرارها). لولا `include='all'` كانت هتقتصر بس على الأعمدة الرقمية.

**ليه استخدمناها:** عشان نفهم **توزيع (distribution)** كل عمود بنظرة واحدة — مثلاً هنا بنلاحظ إن متوسط عمر العملاء 44 سنة تقريبًا، متوسط قيمة الشراء 59.76 دولار، وإن أكتر Category شراءً هي Clothing، وإن أكتر Location هي Montana. وده كمان بيساعد نكتشف أي outliers غريبة بدري.

**الغرض:** فهم أعمق للبيانات (statistical profiling) قبل اتخاذ أي قرار تنظيف أو هندسة خصائص (feature engineering).

In [5]:
# Checking if missing data or null values are present in the dataset

df.isnull().sum()

Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64

**الشرح:** `df.isnull().sum()` بترجع عدد القيم الفارغة (`NaN`) في كل عمود على حدة.

**ليه استخدمناها:** بعد ما شفنا في `df.info()` إن فيه عمود فيه نقص، هنا بنتأكد بالظبط بكام صف ناقص، وطلع إن `Review Rating` فيها 37 قيمة فاضية، وعمودين `Unnamed: 18` و`Unnamed: 19` كلهم فاضيين (3900 من أصل 3900).

**الغرض:** تحديد المشكلة بدقة (exact missing count) عشان نقرر الاستراتيجية المناسبة للتعامل معاها — نعوض (impute)، نمسح الصف، أو نمسح العمود كله.

In [6]:
# Imputing missing values in Review Rating column with the median rating of the product category

df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))

**الشرح:** بنستخدم `groupby('Category')` عشان نقسّم الداتا حسب فئة المنتج (Clothing, Footwear, Outerwear, Accessories)، وبعدين `transform(lambda x: x.fillna(x.median()))` بتعوض أي قيمة فاضية في `Review Rating` بالـ **median** الخاص بنفس الفئة (مش median عام لكل الداتا).

**ليه استخدمناها (Why median بالتحديد؟):**
- استخدمنا **median** مش **mean** لأن الـ median أكتر مقاومة (robust) للـ outliers ومناسب أكتر لتقييمات (ratings) بتتراوح في مدى محدود (2.5 - 5.0).
- استخدمنا `groupby` بدل تعويض عام (global fillna) عشان التقييم المتوقع لمنتج Footwear مثلاً يكون قريب من تقييمات باقي منتجات الـ Footwear، مش رقم عشوائي من كل الداتا.

**الغرض:** الحفاظ على أكبر قدر من البيانات (بدل مسح الصفوف الناقصة) مع تقليل التحيّز (bias) الناتج عن التعويض.

In [7]:
df.isnull().sum()

Customer ID               0
Age                       0
Gender                    0
Item Purchased            0
Category                  0
Purchase Amount (USD)     0
Location                  0
Size                      0
Color                     0
Season                    0
Review Rating             0
Subscription Status       0
Shipping Type             0
Discount Applied          0
Promo Code Used           0
Previous Purchases        0
Payment Method            0
Frequency of Purchases    0
dtype: int64

**الشرح:** إعادة تشغيل `df.isnull().sum()` عشان نتأكد إن عملية التعويض (imputation) اللي عملناها فوق نجحت فعلاً.

**ليه استخدمناها:** ده مبدأ مهم في أي pipeline تنظيف بيانات: **تحقّق بعد كل تعديل (verify after every change)**. لاحظ إن `Review Rating` بقت 0 (اتصلحت)، لكن `Unnamed: 18` و`Unnamed: 19` لسه فاضيين — وده متوقع لأننا لسه ما اتعاملناش معاهم.

**الغرض:** ضمان جودة الخطوة اللي فاتت قبل ما نكمل في باقي التنظيف.

In [8]:
# Renaming columns according to snake casing for better readability and documentation

df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(' ','_')
df = df.rename(columns={'purchase_amount_(usd)':'purchase_amount'})

**الشرح:** بنعمل standardize لأسماء الأعمدة بـ 3 خطوات: (1) `str.lower()` تحويل كل الحروف لـ lowercase، (2) `str.replace(' ', '_')` استبدال المسافات بـ underscore، (3) `rename()` لتصحيح اسم عمود معين (`purchase_amount_(usd)` → `purchase_amount`) عشان الأقواس ممكن تسبب مشاكل في بعض الأدوات (زي SQL).

**ليه استخدمناها:** الأسماء الأصلية زي `"Purchase Amount (USD)"` فيها مسافات وأحرف كبيرة وأقواس، وده بيخلي الكتابة تعبانة وعرضة للأخطاء (لازم تكتبها بالكامل جوه `[]` بدل `df.column_name`)، وممكن كمان تعمل مشاكل لو رفعناها لقاعدة بيانات SQL.

**الغرض:** توحيد أسلوب تسمية الأعمدة (naming convention) بصيغة `snake_case` القياسية في بايثون، عشان الكود يبقى أسهل قراءة وأسهل صيانة (maintainability) وجاهز أكتر للتعامل مع SQL.

In [9]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='str')

**الشرح:** طباعة `df.columns` عشان نشوف أسماء الأعمدة بعد إعادة التسمية.

**ليه استخدمناها:** خطوة تحقق بسيطة (sanity check) للتأكد إن كل الأسماء اتحولت فعلاً لصيغة `snake_case` صح، وإن مفيش اسم عمود اتكسر أو اتكرر بالغلط.

**الغرض:** التأكد قبل الاستمرار في باقي الخطوات.

In [10]:
# create a new column age_group
labels = ['Young Adult', 'Adult', 'Middle-aged', 'Senior']
df['age_group'] = pd.qcut(df['age'], q=4, labels = labels)

**الشرح:** بنعمل عمود جديد اسمه `age_group` باستخدام `pd.qcut()`، وهي دالة بتقسم البيانات لعدد معين من المجموعات (هنا 4 quantiles) بحيث كل مجموعة يكون فيها **نفس العدد تقريبًا من الصفوف** (مش نفس المدى العددي زي `pd.cut`). سمّينا المجموعات: Young Adult, Adult, Middle-aged, Senior.

**ليه استخدمناها (qcut مش cut):** `cut` بيقسم حسب المدى (range) بس، فلو الأعمار متكدسة في نطاق معين هتلاقي فئة فيها ناس كتير جدًا وفئة تانية فاضية تقريبًا. `qcut` بيحل المشكلة دي لأنه بيوزع الأشخاص بالتساوي على الفئات، وده مفيد جدًا في التحليل والـ visualizations عشان كل فئة تبقى ممثلة كفاية إحصائيًا.

**الغرض:** تحويل عمر رقمي مستمر (continuous) لفئة تصنيفية (categorical) — ده بيسهل تحليل سلوك الشراء حسب الفئة العمرية، ومفيد جدًا كـ feature في نماذج churn prediction أو customer segmentation بدل التعامل مع كل عمر رقم لوحده.

In [11]:
df[['age','age_group']].head(10)

,age,age_group
0,55,Middle-aged
1,19,Young Adult
2,50,Middle-aged
3,21,Young Adult
4,45,Middle-aged
5,46,Middle-aged
6,63,Senior
7,27,Young Adult
8,26,Young Adult
9,57,Middle-aged


**الشرح:** بنعرض عمودي `age` و`age_group` مع بعض لأول 10 صفوف.

**ليه استخدمناها:** خطوة تحقق بصرية (visual sanity check) عشان نتأكد إن التقسيم (binning) اللي عمله `qcut` منطقي — مثلاً عمر 19 اتحط في "Young Adult" وعمر 63 اتحط في "Senior"، وده متسق مع المتوقع.

**الغرض:** التأكد من صحة الـ feature الجديدة قبل الاعتماد عليها في أي تحليل تاني.

In [12]:
# create new column purchase_frequency_days

frequency_mapping = {
    'Fortnightly': 14,
    'Weekly': 7,
    'Monthly': 30,
    'Quarterly': 90,
    'Bi-Weekly': 14,
    'Annually': 365,
    'Every 3 Months': 90
}

df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)

**الشرح:** بنعمل `dictionary` اسمه `frequency_mapping` بيربط كل قيمة نصية في عمود `frequency_of_purchases` (زي "Weekly", "Annually") بعدد أيام تقريبي مقابلها (7, 365, ...). بعدين بنستخدم `.map()` عشان نطبق التحويل ده على العمود كله وننشئ عمود جديد `purchase_frequency_days`.

**ليه استخدمناها:** عمود `frequency_of_purchases` هو نص (categorical) والموديلات الإحصائية وموديلات الـ machine learning بشكل عام بتتعامل أحسن مع أرقام (numeric)، وكمان الأرقام (زي عدد الأيام) بتديك مقياس (scale) واضح تقدر تقارن بيه: مثلاً تعرف إن "Weekly" (7 أيام) أكتر تكرارًا بكتير من "Annually" (365 يوم) بشكل رقمي مباشر.

**الغرض:** تحويل feature نصي لرقمي (encoding) بطريقة بتحافظ على المعنى الحقيقي (عدد الأيام الفعلي)، وده مهم جدًا في مجال بحثك عن churn prediction لأن "معدل الشراء بالأيام" ممكن يكون مؤشر قوي على نشاط العميل أو خموله.

In [13]:
df[['purchase_frequency_days','frequency_of_purchases']].head(10)

,purchase_frequency_days,frequency_of_purchases
0,14,Fortnightly
1,14,Fortnightly
2,7,Weekly
3,7,Weekly
4,365,Annually
5,7,Weekly
6,90,Quarterly
7,7,Weekly
8,365,Annually
9,90,Quarterly


**الشرح:** عرض عمودي `purchase_frequency_days` و`frequency_of_purchases` جنب بعض لأول 10 صفوف.

**ليه استخدمناها:** تحقق سريع إن الـ `mapping` اتطبق صح — مثلاً "Fortnightly" ✓ اتحولت لـ 14 يوم، و"Weekly" ✓ اتحولت لـ 7 أيام، و"Annually" ✓ اتحولت لـ 365 يوم.

**الغرض:** ضمان صحة الـ encoding قبل الاعتماد عليه.

In [14]:
df[['discount_applied','promo_code_used']].head(10)

,discount_applied,promo_code_used
0,Yes,Yes
1,Yes,Yes
2,Yes,Yes
3,Yes,Yes
4,Yes,Yes
5,Yes,Yes
6,Yes,Yes
7,Yes,Yes
8,Yes,Yes
9,Yes,Yes


**الشرح:** بنعرض عمودي `discount_applied` و`promo_code_used` مع بعض لأول 10 صفوف.

**ليه استخدمناها:** لاحظنا إن القيم بتتكرر بنفس الشكل صف بصف (كل ما يكون Yes في الأول، يبقى Yes في التاني)، فده بيثير شك إن العمودين ممكن يكونوا **نفس المعلومة بالظبط** (duplicate feature)، فقررنا نتحقق من الفرضية دي بشكل رسمي في الخلية اللي جاية.

**الغرض:** اكتشاف تكرار محتمل في البيانات (feature redundancy) قبل ما نستخدمها في أي تحليل أو نموذج.

In [15]:
(df['discount_applied'] == df['promo_code_used']).all()

np.True_

**الشرح:** `(df['discount_applied'] == df['promo_code_used']).all()` بتعمل مقارنة (element-wise comparison) بين العمودين صف بصف، وبعدين `.all()` بترجع `True` لو **كل** الصفوف متطابقة، أو `False` لو فيه ولو صف واحد مختلف.

**ليه استخدمناها:** دي طريقة برمجية دقيقة (rigorous) للتأكد من فرضية "العمودين متطابقين" بدل ما نعتمد بس على النظر بالعين لأول 10 صفوف. النتيجة طلعت `True` يعني فعلاً العمودين متطابقين 100% في كل الـ 3900 صف.

**الغرض:** تأكيد رسمي (statistical confirmation) قبل اتخاذ قرار حذف أحد العمودين.

In [16]:
# Dropping promo code used column

df = df.drop('promo_code_used', axis=1)

**الشرح:** بنستخدم `df.drop('promo_code_used', axis=1)` عشان نمسح عمود `promo_code_used` بالكامل من الـ DataFrame (`axis=1` معناها إننا بنمسح عمود مش صف).

**ليه استخدمناها:** بما إننا أكدنا في الخلية اللي فاتت إن `promo_code_used` بيحمل بالظبط نفس معلومة `discount_applied`، فمفيش داعي نسيبه — ده بيعتبر **redundant feature**. الاحتفاظ بيه ممكن يسبب مشاكل زي الـ **multicollinearity** لو استخدمنا الداتا بعدين في نموذج إحصائي أو machine learning.

**الغرض:** تقليل حجم البيانات وتبسيطها (dimensionality reduction بسيط) من غير ما نفقد أي معلومة فعلية.

In [17]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchase_frequency_days'],
      dtype='str')

**الشرح:** طباعة `df.columns` تاني عشان نشوف القائمة النهائية للأعمدة بعد كل عمليات التنظيف والإضافة (rename, impute, drop, و الـ features الجديدة `age_group` و`purchase_frequency_days`).

**ليه استخدمناها:** خطوة مراجعة نهائية (final review) قبل الانتقال لمرحلة تصدير البيانات.

**الغرض:** التأكد إن الـ DataFrame وصل للشكل النهائي المطلوب قبل رفعه لقاعدة البيانات.

> ⚠️ **ملحوظة:** لسه باقي عمودين `unnamed:_18` و`unnamed:_19` في القائمة وهما فاضيين بالكامل (كل القيم NaN) — كان ممكن نضيف خطوة `df = df.drop(columns=['unnamed:_18','unnamed:_19'])` لحذفهم زي ما عملنا مع `promo_code_used`، بس النوتبوك الأصلي سابهم من غير ما يتم التعامل معاهم.

## Code for MS SQL Server

**الشرح:** من هنا وطالع، الجزء ده مخصص لتصدير الـ DataFrame النضيف لقاعدة بيانات **MS SQL Server** بدل ما يفضل بس ملف CSV محلي.

**ليه بنعمل كده:** رفع الداتا لقاعدة بيانات بيدي مميزات زي: إمكانية الوصول ليها من أدوات تانية (Power BI, Excel, أدوات SQL)، تخزين أكثر تنظيمًا وأمانًا، وإمكانية عمل queries عليها مباشرة من غير ما تفتح بايثون كل مرة.

**الغرض:** جعل النسخة النضيفة من البيانات متاحة ومركزية (centralized) لأي استخدام لاحق.

In [18]:
!pip install pyodbc sqlalchemy 


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


**الشرح:** بنستخدم `!pip install` (علامة `!` بتخلي الكود يتنفذ كـ shell command جوه الـ notebook) عشان نثبت مكتبتين:
- **`pyodbc`**: المكتبة اللي بتعمل الاتصال الفعلي (low-level driver connection) مع قواعد بيانات زي SQL Server عن طريق ODBC.
- **`sqlalchemy`**: مكتبة أعلى مستوى (toolkit/ORM) بتسهّل التعامل مع قواعد البيانات بشكل عام من بايثون، ومتوافقة مع `pandas` (زي دالة `to_sql` و`read_sql`).

**ليه استخدمناها:** من غير المكتبتين دول، `pandas` مش هتقدر "تتكلم" مع SQL Server مباشرة. `sqlalchemy` هي اللي بتوفر الـ `engine`، و`pyodbc` هي اللي شغالة "تحت السطح" كـ driver فعلي للاتصال.

**الغرض:** تجهيز البيئة (environment setup) قبل خطوة الاتصال الفعلي بقاعدة البيانات.

In [19]:
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus

# اسم السيرفر والداتابيز بناءً على الصورة
server = "M_M_SOBHY"
database = "customer_behavior"

# إعداد تعريف الدرايفر والتوصيل بـ Windows Authentication
driver = quote_plus("ODBC Driver 17 for SQL Server")
connection_string = (
    f"mssql+pyodbc://@{server}/{database}?driver={driver}&trusted_connection=yes"
)

# إنشاء المحرك (Engine)
engine = create_engine(connection_string)

# كتابة الـ DataFrame إلى SQL Server
df.to_sql("customer", engine, if_exists="replace", index=False)

# قراءة أول 5 صفوف للتأكد
result = pd.read_sql("SELECT TOP 5 * FROM customer;", engine)
print(result)

   customer_id  age gender item_purchased  category  purchase_amount  \
0            1   55   Male         Blouse  Clothing               53   
1            2   19   Male        Sweater  Clothing               64   
2            3   50   Male          Jeans  Clothing               73   
3            4   21   Male        Sandals  Footwear               90   
4            5   45   Male         Blouse  Clothing               49   

        location size      color  season  review_rating subscription_status  \
0       Kentucky    L       Gray  Winter            3.1                 Yes   
1          Maine    L     Maroon  Winter            3.1                 Yes   
2  Massachusetts    S     Maroon  Spring            3.1                 Yes   
3   Rhode Island    M     Maroon  Spring            3.5                 Yes   
4         Oregon    M  Turquoise  Spring            2.7                 Yes   

   shipping_type discount_applied  previous_purchases payment_method  \
0        Express    

**الشرح خطوة بخطوة:**

1. **`server` و `database`** — تحديد اسم السيرفر المحلي (`M_M_SOBHY`) واسم قاعدة البيانات (`customer_behavior`) اللي هنكتب فيها.
2. **`driver = quote_plus("ODBC Driver 17 for SQL Server")`** — بنجهز اسم الـ ODBC driver ونعمله `quote_plus` (URL-encoding) عشان أي مسافات أو رموز خاصة في الاسم متسببش مشكلة في الـ connection string.
3. **`connection_string`** — بنبني نص الاتصال (connection string) اللي فيه كل المعلومات اللازمة للاتصال: نوع القاعدة (`mssql`)، الـ driver (`pyodbc`)، السيرفر، والداتابيز.
4. **`trusted_connection=yes`** — ده معناه إننا بنستخدم **Windows Authentication** (بيانات اعتماد نظام التشغيل بتاعتك) بدل ما نكتب username/password بشكل صريح. ده مناسب هنا لأن السيرفر شغال محليًا على نفس الجهاز.
5. **`create_engine(connection_string)`** — بنستخدم `sqlalchemy` عشان نبني الـ **Engine**، وهو الكائن (object) المسؤول عن إدارة الاتصال الفعلي وتنفيذ الأوامر على قاعدة البيانات.
6. **`df.to_sql("customer", engine, if_exists="replace", index=False)`** — بنرفع الـ DataFrame كامل كجدول اسمه `customer` في قاعدة البيانات. `if_exists="replace"` معناها لو الجدول موجود بالفعل هيتمسح ويتعمل تاني من الصفر (مفيد وقت التطوير، بس لازم تكون حذر منه في بيئة production عشان ممكن يمسح بيانات مهمة!). `index=False` عشان منضفش عمود إضافي بيمثل index بايثون جوه الجدول.
7. **`pd.read_sql("SELECT TOP 5 * FROM customer;", engine)`** — أخيرًا بنقرا أول 5 صفوف من الجدول اللي رفعناه، كخطوة تحقق نهائية (final verification) إن كل حاجة اتخزنت صح.

**⚠️ ملحوظة أمان مهمة:** استخدام `trusted_connection=yes` كويس ومناسب لما تكون شغال على جهازك الشخصي (local development)، لكن لو هتنقل الكود ده لسيرفر حقيقي أو بيئة إنتاج (production)، الأفضل تستخدم بيانات اعتماد مُدارة بأمان (secure credentials / environment variables) بدل الاعتماد الكامل على هوية نظام التشغيل.

**الغرض من الخلية دي بالكامل:** إغلاق دورة حياة البيانات (data lifecycle) في النوتبوك: من تحميل CSV خام → تنظيف وهندسة خصائص → تخزين نهائي في قاعدة بيانات علائقية (relational database) جاهزة لأي استخدام أو تحليل لاحق (Power BI, تقارير SQL، أو حتى قراءتها تاني في بايثون لعمل الـ modeling بتاع الـ churn prediction).